## Fraud Detection Model Setup

### Purpose
This notebook trains a **fraud detection model**, registers it in **Unity Catalog**, and deploys it to a **Model Serving endpoint**. Once the endpoint is live, the consumer notebook uses `ai_query()` to score transactions in real time.

### What Gets Created
| Resource | Location |
|---|---|
| MLflow experiment | Auto-created in this notebook’s context |
| Registered model | `fintech_demo.models.fraud_scorer` |
| Serving endpoint | `fintech-fraud-scoring` |

### Run This First
Run all cells top-to-bottom **before** starting the consumer notebook’s Step 7. The endpoint takes 5–10 minutes to provision.

> **Note:** This notebook should be run **after** the Producer notebook has created the `fintech_demo` catalog (Step 0 in the Producer creates all base resources).

In [0]:
# ---------------------------------------------------------------------------
# Shared configuration — must match the producer/consumer notebooks
# ---------------------------------------------------------------------------
CATALOG        = "fintech_demo"
MODELS_SCHEMA  = "models"
MODEL_NAME     = f"{CATALOG}.{MODELS_SCHEMA}.fraud_scorer"
ENDPOINT_NAME  = "fintech-fraud-scoring"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{MODELS_SCHEMA}")

print(f"\u2705 Models schema : {CATALOG}.{MODELS_SCHEMA}")
print(f"\u2705 Model name    : {MODEL_NAME}")
print(f"\u2705 Endpoint name : {ENDPOINT_NAME}")

In [0]:
# ---------------------------------------------------------------------------
# Generate synthetic training data with realistic fraud patterns
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd

np.random.seed(42)
N = 10_000

# Features matching the Silver table schema
amount       = np.random.exponential(scale=500, size=N).clip(0.50, 9999.99)
hour_of_day  = np.random.randint(0, 24, size=N).astype(float)
day_of_week  = np.random.randint(1, 8, size=N).astype(float)    # 1=Sun, 7=Sat
is_high_value = (amount > 5000).astype(float)
is_off_hours  = ((hour_of_day < 6) | (hour_of_day > 22)).astype(float)

# Fraud labels — rule-based with noise to create learnable patterns
# Base fraud rate ~5%, increases with risk signals
fraud_prob = (
    0.03                                         # baseline
    + 0.12 * is_off_hours                        # off-hours transactions
    + 0.08 * is_high_value                       # high-value transactions
    + 0.25 * (is_off_hours * is_high_value)      # both = very suspicious
    + 0.05 * (hour_of_day < 4).astype(float)     # deep night (midnight-4am)
    + 0.02 * (amount / 10000)                    # amount scales risk
)
fraud_prob = fraud_prob.clip(0, 0.95)
is_fraud = (np.random.random(N) < fraud_prob).astype(int)

# Assemble DataFrame
train_df = pd.DataFrame({
    "amount":        amount,
    "hour_of_day":   hour_of_day,
    "day_of_week":   day_of_week,
    "is_high_value": is_high_value,
    "is_off_hours":  is_off_hours,
})

print(f"\U0001F4CA Training data: {len(train_df):,} rows")
print(f"   Fraud rate: {is_fraud.mean():.1%}")
print(f"   Features:   {list(train_df.columns)}")
train_df.head()

In [0]:
# ---------------------------------------------------------------------------
# Train a RandomForest fraud scorer, wrap in PyFunc, register in UC
# ---------------------------------------------------------------------------
import mlflow
import pickle
import tempfile
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    train_df, is_fraud, test_size=0.2, random_state=42, stratify=is_fraud
)

# Train
clf = RandomForestClassifier(
    n_estimators=100, max_depth=8, class_weight="balanced", random_state=42
)
clf.fit(X_train, y_train)

# Evaluate
y_proba = clf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"\U0001F3AF Test AUC-ROC: {auc:.4f}")
print(classification_report(y_test, (y_proba > 0.5).astype(int), target_names=["legit", "fraud"]))

# Save sklearn model to temp file for PyFunc artifact
with tempfile.NamedTemporaryFile(suffix=".pkl", delete=False) as f:
    pickle.dump(clf, f)
    model_artifact_path = f.name

# Define PyFunc that returns fraud probability (not just class label)
class FraudScorer(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import pickle
        with open(context.artifacts["sklearn_model"], "rb") as f:
            self.model = pickle.load(f)

    def predict(self, context, model_input, params=None):
        import pandas as pd
        proba = self.model.predict_proba(model_input)[:, 1]
        return proba

# Infer signature from real data
sample_input = X_test.head(5)
sample_output = clf.predict_proba(X_test.head(5))[:, 1]
signature = infer_signature(sample_input, sample_output)

# Log and register
with mlflow.start_run(run_name="fraud_scorer_rf") as run:
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 8)

    model_info = mlflow.pyfunc.log_model(
        artifact_path="fraud_scorer",
        python_model=FraudScorer(),
        artifacts={"sklearn_model": model_artifact_path},
        signature=signature,
        input_example=sample_input,
        pip_requirements=["scikit-learn", "pandas", "numpy"],
        registered_model_name=MODEL_NAME,
    )

print(f"\n\u2705 Model registered: {MODEL_NAME}")
print(f"   Model URI: {model_info.model_uri}")
print(f"   Run ID:    {run.info.run_id}")

In [0]:
# ---------------------------------------------------------------------------
# Validate: load the registered model and score a test batch
# ---------------------------------------------------------------------------
import mlflow

loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

test_cases = pd.DataFrame({
    "amount":        [50.0, 8500.0, 200.0, 7000.0],
    "hour_of_day":   [14.0, 3.0, 10.0, 2.0],
    "day_of_week":   [3.0, 1.0, 5.0, 7.0],
    "is_high_value": [0.0, 1.0, 0.0, 1.0],
    "is_off_hours":  [0.0, 1.0, 0.0, 1.0],
})

scores = loaded_model.predict(test_cases)

print("\U0001F9EA Local validation results:")
for i, row in test_cases.iterrows():
    risk = "\U0001F534 HIGH" if scores[i] > 0.5 else "\U0001F7E1 MED" if scores[i] > 0.2 else "\U0001F7E2 LOW"
    print(f"   ${row['amount']:>8,.2f}  hr={int(row['hour_of_day']):>2}  "
          f"high_val={int(row['is_high_value'])}  off_hrs={int(row['is_off_hours'])}  "
          f"\u2192 score={scores[i]:.4f}  {risk}")

print(f"\n\u2705 Model validation passed")

In [0]:
# ---------------------------------------------------------------------------
# Deploy the model to a Serving Endpoint
# ---------------------------------------------------------------------------
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput, ServedEntityInput
)

w = WorkspaceClient()

# Check if endpoint already exists
try:
    existing = w.serving_endpoints.get(ENDPOINT_NAME)
    print(f"\u2139\ufe0f  Endpoint '{ENDPOINT_NAME}' already exists (state: {existing.state.ready})")
    print(f"   Updating to latest model version...")

    # Get latest model version
    from databricks.sdk.service.catalog import ModelVersionInfoStatus
    versions = list(w.model_versions.list(full_name=MODEL_NAME))
    latest_version = str(max(int(v.version) for v in versions))

    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=[
            ServedEntityInput(
                entity_name=MODEL_NAME,
                entity_version=latest_version,
                scale_to_zero_enabled=True,
                workload_size="Small",
            )
        ]
    )
    print(f"\u2705 Endpoint updated to model version {latest_version}")

except Exception as e:
    if "RESOURCE_DOES_NOT_EXIST" in str(e) or "does not exist" in str(e).lower():
        print(f"\U0001F680 Creating endpoint '{ENDPOINT_NAME}'...")
        print(f"   This takes 5\u201310 minutes. Please wait.\n")

        w.serving_endpoints.create_and_wait(
            name=ENDPOINT_NAME,
            config=EndpointCoreConfigInput(
                served_entities=[
                    ServedEntityInput(
                        entity_name=MODEL_NAME,
                        entity_version="1",
                        scale_to_zero_enabled=True,
                        workload_size="Small",
                    )
                ]
            )
        )
        print(f"\u2705 Endpoint '{ENDPOINT_NAME}' is READY")
    else:
        raise

In [0]:
# ---------------------------------------------------------------------------
# Test the live endpoint with a sample request
# ---------------------------------------------------------------------------
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Test with a suspicious transaction (high value, off hours)
response = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=[
        {"amount": 8500.0, "hour_of_day": 3.0, "day_of_week": 1.0,
         "is_high_value": 1.0, "is_off_hours": 1.0},
        {"amount": 25.0, "hour_of_day": 14.0, "day_of_week": 3.0,
         "is_high_value": 0.0, "is_off_hours": 0.0},
    ]
)

print(f"\U0001F310 Live endpoint test results:")
print(f"   Suspicious txn ($8,500 at 3am): {response.predictions[0]:.4f}")
print(f"   Normal txn ($25 at 2pm):        {response.predictions[1]:.4f}")
print(f"\n\u2705 Endpoint is live and scoring correctly!")
print(f"\n\u27a1\ufe0f  Now switch to the consumer notebook and run Step 7 (ai_query fraud scoring)")

### Endpoint is Ready!

The `fintech-fraud-scoring` endpoint is now live. Head over to the **consumer notebook** and run **Step 7** to score Silver transactions using `ai_query()`.

> **Cleanup:** To delete the endpoint when done, run the cleanup cell in the consumer notebook (Step 8). The endpoint should also be manually deleted from the Serving UI if the catalog cascade drop doesn’t remove it.